# Superstore Sales — SQL Analysis
**Dataset:** Sample - Superstore.csv (9,994 rows, 21 original columns)  
**Objective:** Filter, aggregate, and derive business insights using SQL queries on a retail sales dataset.  
**Tools:** Python (pandas, sqlite3) for loading; raw SQL for all analysis.

---
## Step 1 — Load Dataset into SQLite

## Step 1 - Loading the Data

First I loaded the superstore CSV file into a SQLite database so I could run SQL queries on it.

One problem I ran into was a UnicodeDecodeError when reading the file. After checking I found 
the file uses latin1 encoding so I added that parameter to fix it.

I also added 5 new columns before loading because I felt the raw data was missing some useful 
business metrics:
- Profit_Margin_Pct - to see actual profitability not just raw profit numbers
- Revenue_per_Unit - average selling price per item
- Ship_Delay_Days - to check if shipping speed affects anything
- Profit_Flag - simple label to quickly filter loss making orders
- Discount_Band - to group discounts and see their impact on profit

In [1]:
import pandas as pd
import sqlite3

df = pd.read_csv('Sample - Superstore.csv', encoding='latin1')

# Normalize column names for SQL compatibility
df.columns = [c.strip().replace(' ', '_').replace('-', '_') for c in df.columns]

df['Order_Date'] = pd.to_datetime(df['Order_Date'])
df['Ship_Date']  = pd.to_datetime(df['Ship_Date'])

# ── Engineered columns (unique additions) ──────────────────────────────────
# 1. Profit margin as a percentage of sales
df['Profit_Margin_Pct'] = ((df['Profit'] / df['Sales']) * 100).round(2)

# 2. Revenue earned per unit sold
df['Revenue_per_Unit'] = (df['Sales'] / df['Quantity']).round(2)

# 3. Days taken to ship after the order was placed
df['Ship_Delay_Days'] = (df['Ship_Date'] - df['Order_Date']).dt.days

# 4. Binary profitability label — useful for quick filtering
df['Profit_Flag'] = df['Profit'].apply(lambda x: 'Profitable' if x > 0 else 'Loss')

# 5. Discount bucketed into bands for cohort analysis
df['Discount_Band'] = pd.cut(
    df['Discount'],
    bins=[-0.01, 0, 0.2, 0.4, 1.0],
    labels=['No Discount', 'Low', 'Medium', 'High']
)

# Load into SQLite
con = sqlite3.connect('superstore.db')
df.to_sql('superstore', con, if_exists='replace', index=False)

print(f'Loaded {len(df):,} rows and {len(df.columns)} columns into superstore.db')
print('New columns added:', ['Profit_Margin_Pct', 'Revenue_per_Unit', 'Ship_Delay_Days', 'Profit_Flag', 'Discount_Band'])

Loaded 9,994 rows and 26 columns into superstore.db
New columns added: ['Profit_Margin_Pct', 'Revenue_per_Unit', 'Ship_Delay_Days', 'Profit_Flag', 'Discount_Band']


---
## Step 2 — Explore the Table (Schema + Sample Data)

## Step 2 - Exploring the Table

Before writing any queries I wanted to understand the structure of the data.
So I checked the schema to see column names and data types, looked at a few 
sample rows, and confirmed the total row count matches the original CSV.

In [2]:
# 2.1 — Schema
schema = pd.read_sql_query("PRAGMA table_info(superstore);", con)
print(schema[['cid', 'name', 'type']].to_string(index=False))

 cid              name      type
   0            Row_ID   INTEGER
   1          Order_ID      TEXT
   2        Order_Date TIMESTAMP
   3         Ship_Date TIMESTAMP
   4         Ship_Mode      TEXT
   5       Customer_ID      TEXT
   6     Customer_Name      TEXT
   7           Segment      TEXT
   8           Country      TEXT
   9              City      TEXT
  10             State      TEXT
  11       Postal_Code   INTEGER
  12            Region      TEXT
  13        Product_ID      TEXT
  14          Category      TEXT
  15      Sub_Category      TEXT
  16      Product_Name      TEXT
  17             Sales      REAL
  18          Quantity   INTEGER
  19          Discount      REAL
  20            Profit      REAL
  21 Profit_Margin_Pct      REAL
  22  Revenue_per_Unit      REAL
  23   Ship_Delay_Days   INTEGER
  24       Profit_Flag      TEXT
  25     Discount_Band      TEXT


**Output:**
```
 cid              name      type
   0            Row_ID   INTEGER
   1          Order_ID      TEXT
   2        Order_Date TIMESTAMP
   3         Ship_Date TIMESTAMP
   4         Ship_Mode      TEXT
   5       Customer_ID      TEXT
   6     Customer_Name      TEXT
   7           Segment      TEXT
   8           Country      TEXT
   9              City      TEXT
  10             State      TEXT
  11       Postal_Code   INTEGER
  12            Region      TEXT
  13        Product_ID      TEXT
  14          Category      TEXT
  15      Sub_Category      TEXT
  16      Product_Name      TEXT
  17             Sales      REAL
  18          Quantity   INTEGER
  19          Discount      REAL
  20            Profit      REAL
  21 Profit_Margin_Pct      REAL
  22  Revenue_per_Unit      REAL
  23   Ship_Delay_Days   INTEGER
  24       Profit_Flag      TEXT
  25     Discount_Band      TEXT
```

In [3]:
# 2.2 — Sample rows
sample_q = """
SELECT Row_ID, Order_ID, Order_Date, Customer_Name,
       Category, Region, Sales, Profit,
       Profit_Margin_Pct, Ship_Delay_Days
FROM   superstore
LIMIT  5;
"""
pd.read_sql_query(sample_q, con)

,Row_ID,Order_ID,Order_Date,Customer_Name,Category,Region,Sales,Profit,Profit_Margin_Pct,Ship_Delay_Days
0,1,CA-2016-152156,2016-11-08 00:00:00,Claire Gute,Furniture,South,261.9600,41.9136,16.00,3
1,2,CA-2016-152156,2016-11-08 00:00:00,Claire Gute,Furniture,South,731.9400,219.5820,30.00,3
2,3,CA-2016-138688,2016-06-12 00:00:00,Darrin Van Huff,Office Supplies,West,14.6200,6.8714,47.00,4
3,4,US-2015-108966,2015-10-11 00:00:00,Sean O'Donnell,Furniture,South,957.5775,-383.0310,-40.00,7
4,5,US-2015-108966,2015-10-11 00:00:00,Sean O'Donnell,Office Supplies,South,22.3680,2.5164,11.25,7


**Output:**
```
 Row_ID       Order_ID  Order_Date     Customer_Name        Category Region    Sales    Profit  Profit_Margin_Pct  Ship_Delay_Days
      1 CA-2016-152156  2016-11-08       Claire Gute       Furniture  South  261.960   41.914              16.00                3
      2 CA-2016-152156  2016-11-08       Claire Gute       Furniture  South  731.940  219.582              30.00                3
      3 CA-2016-138688  2016-06-12   Darrin Van Huff Office Supplies   West   14.620    6.871              47.00                4
      4 US-2015-108966  2015-10-11    Sean O'Donnell       Furniture  South  957.578 -383.031             -40.00                7
      5 US-2015-108966  2015-10-11    Sean O'Donnell Office Supplies  South   22.368    2.516              11.25                7
```

In [4]:
# 2.3 — Total rows
pd.read_sql_query("SELECT COUNT(*) AS total_rows FROM superstore;", con)

,total_rows
0,9994


**Output:** `total_rows = 9994`

---
## Step 3 — WHERE Filters

## Step 3 - Filtering the Data

Here I used WHERE clauses to slice the data in different ways.
I filtered by region, category, date range and sales amount to 
see how the data looks for specific conditions. I also used the 
Profit_Flag column I created earlier to quickly find loss making orders.

In [5]:
# 3.1 — Filter by Region: West
q = """
SELECT Row_ID, Customer_Name, Region, Sales, Profit
FROM   superstore
WHERE  Region = 'West'
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Row_ID,Customer_Name,Region,Sales,Profit
0,3,Darrin Van Huff,West,14.620,6.8714
1,6,Brosina Hoffman,West,48.860,14.1694
2,7,Brosina Hoffman,West,7.280,1.9656
3,8,Brosina Hoffman,West,907.152,90.7152
4,9,Brosina Hoffman,West,18.504,5.7825


**Output:**
```
 Row_ID   Customer_Name Region   Sales  Profit
      3 Darrin Van Huff   West  14.620  6.871
      6 Brosina Hoffman   West  48.860 14.169
      7 Brosina Hoffman   West   7.280  1.966
      8 Brosina Hoffman   West 907.152 90.715
      9 Brosina Hoffman   West  18.504  5.783
```

In [6]:
# 3.2 — Filter by Category + High Discount
q = """
SELECT Row_ID, Product_Name, Category, Sales, Discount
FROM   superstore
WHERE  Category = 'Technology'
  AND  Discount > 0.2
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Row_ID,Product_Name,Category,Sales,Discount
0,131,Anker 36W 4-Port USB Wall Charger Travel Power...,Technology,59.970,0.4
1,166,Lexmark MX611dhe Monochrome Laser Printer,Technology,8159.952,0.4
2,215,Speck Products Candyshell Flip Case,Technology,62.982,0.4
3,216,Cisco 9971 IP Video Phone Charcoal,Technology,1188.000,0.7
4,224,Swingline SM12-08 MicroCut Jam Free Shredder,Technology,479.988,0.7


**Output:**
```
 Row_ID                                         Product_Name   Category    Sales  Discount
    131 Anker 36W 4-Port USB Wall Charger Travel Power Adapter Technology   59.970       0.4
    166             Lexmark MX611dhe Monochrome Laser Printer Technology 8159.952       0.4
    215                     Speck Products Candyshell Flip Case Technology   62.982       0.4
    216                    Cisco 9971 IP Video Phone Charcoal Technology 1188.000       0.7
    224          Swingline SM12-08 MicroCut Jam Free Shredder Technology  479.988       0.7
```

In [7]:
# 3.3 — Filter by Date Range: Jan–Jun 2017
q = """
SELECT Order_ID, Order_Date, Customer_Name, Sales
FROM   superstore
WHERE  Order_Date BETWEEN '2017-01-01' AND '2017-06-30'
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Order_ID,Order_Date,Customer_Name,Sales
0,CA-2017-114412,2017-04-15 00:00:00,Andrew Allen,15.552
1,CA-2017-140088,2017-05-28 00:00:00,Patrick O'Donnell,301.960
2,CA-2017-157833,2017-06-17 00:00:00,Katherine Ducich,51.312
3,US-2017-164147,2017-02-02 00:00:00,Dorothy Wardle,59.970
4,US-2017-164147,2017-02-02 00:00:00,Dorothy Wardle,78.304


**Output:**
```
      Order_ID  Order_Date     Customer_Name   Sales
CA-2017-114412  2017-04-15      Andrew Allen  15.552
CA-2017-140088  2017-05-28 Patrick O'Donnell 301.960
CA-2017-157833  2017-06-17  Katherine Ducich  51.312
US-2017-164147  2017-02-02    Dorothy Wardle  59.970
US-2017-164147  2017-02-02    Dorothy Wardle  78.304
```

In [8]:
# 3.4 — High-Value Orders (Sales > $1,000)
q = """
SELECT Customer_Name, Product_Name, Sales, Profit
FROM   superstore
WHERE  Sales > 1000
ORDER  BY Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Customer_Name,Product_Name,Sales,Profit
0,Sean Miller,Cisco TelePresence System EX90 Videoconferenci...,22638.480,-1811.0784
1,Tamara Chand,Canon imageCLASS 2200 Advanced Copier,17499.950,8399.9760
2,Raymond Buch,Canon imageCLASS 2200 Advanced Copier,13999.960,6719.9808
3,Tom Ashbrook,Canon imageCLASS 2200 Advanced Copier,11199.968,3919.9888
4,Hunter Lopez,Canon imageCLASS 2200 Advanced Copier,10499.970,5039.9856


**Output:**
```
Customer_Name                                          Product_Name     Sales     Profit
  Sean Miller Cisco TelePresence System EX90 Videoconferencing Unit 22638.480 -1811.078
 Tamara Chand                 Canon imageCLASS 2200 Advanced Copier 17499.950  8399.976
 Raymond Buch                 Canon imageCLASS 2200 Advanced Copier 13999.960  6719.981
 Tom Ashbrook                 Canon imageCLASS 2200 Advanced Copier 11199.968  3919.989
 Hunter Lopez                 Canon imageCLASS 2200 Advanced Copier 10499.970  5039.986
```

> **Note:** Sean Miller's top order generated a loss of $1,811 despite $22K in revenue — a sign of deep discounting on high-ticket items.

In [9]:
# 3.5 — Loss-Making Orders (using engineered Profit_Flag)
q = """
SELECT Order_ID, Customer_Name, Product_Name, Sales, Profit
FROM   superstore
WHERE  Profit_Flag = 'Loss'
ORDER  BY Profit ASC
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Order_ID,Customer_Name,Product_Name,Sales,Profit
0,CA-2016-108196,Cindy Stewart,Cubify CubeX 3D Printer Double Head Print,4499.985,-6599.9780
1,US-2017-168116,Grant Thornton,Cubify CubeX 3D Printer Triple Head Print,7999.980,-3839.9904
2,CA-2014-169019,Luke Foster,GBC DocuBind P400 Electric Binding System,2177.584,-3701.8928
3,CA-2017-134845,Sharelle Roach,Lexmark MX611dhe Monochrome Laser Printer,2549.985,-3399.9800
4,US-2017-122714,Henry Goldwyn,Ibico EPK-21 Electric Binding System,1889.990,-2929.4845


**Output:**
```
      Order_ID  Customer_Name                              Product_Name    Sales      Profit
CA-2016-108196  Cindy Stewart   Cubify CubeX 3D Printer Double Head Print 4499.985 -6599.978
US-2017-168116 Grant Thornton   Cubify CubeX 3D Printer Triple Head Print 7999.980 -3839.990
CA-2014-169019    Luke Foster    GBC DocuBind P400 Electric Binding System 2177.584 -3701.893
CA-2017-134845 Sharelle Roach Lexmark MX611dhe Monochrome Laser Printer   2549.985 -3399.980
US-2017-122714  Henry Goldwyn      Ibico EPK-21 Electric Binding System   1889.990 -2929.485
```

---
## Step 4 — GROUP BY Aggregations

## Step 4 - Group By Aggregations

Here I grouped the data by region, category, sub-category, segment and discount bands 
to find patterns. I wanted to see which areas are making the most sales and which ones 
are actually profitable. The discount band grouping was the most interesting one because 
it shows how discounting is affecting profit directly.

In [10]:
# 4.1 — Sales, Orders, and Average Sale by Region
q = """
SELECT Region,
       ROUND(SUM(Sales), 2)    AS Total_Sales,
       COUNT(*)                AS Num_Orders,
       ROUND(AVG(Sales), 2)    AS Avg_Sales
FROM   superstore
GROUP  BY Region
ORDER  BY Total_Sales DESC;
"""
pd.read_sql_query(q, con)

,Region,Total_Sales,Num_Orders,Avg_Sales
0,West,725457.82,3203,226.49
1,East,678781.24,2848,238.34
2,Central,501239.89,2323,215.77
3,South,391721.91,1620,241.80


**Output:**
```
 Region  Total_Sales  Num_Orders  Avg_Sales
   West    725457.82        3203     226.49
   East    678781.24        2848     238.34
Central    501239.89        2323     215.77
  South    391721.91        1620     241.80
```

In [11]:
# 4.2 — Sales, Profit, and Avg Margin by Category
q = """
SELECT Category,
       ROUND(SUM(Sales), 2)             AS Total_Sales,
       ROUND(SUM(Profit), 2)            AS Total_Profit,
       ROUND(AVG(Profit_Margin_Pct), 2) AS Avg_Margin_Pct
FROM   superstore
GROUP  BY Category
ORDER  BY Total_Sales DESC;
"""
pd.read_sql_query(q, con)

,Category,Total_Sales,Total_Profit,Avg_Margin_Pct
0,Technology,836154.03,145454.95,15.61
1,Furniture,741999.80,18451.27,3.88
2,Office Supplies,719047.03,122490.80,13.80


**Output:**
```
        Category  Total_Sales  Total_Profit  Avg_Margin_Pct
      Technology    836154.03     145454.95           15.61
       Furniture    741999.80      18451.27            3.88
 Office Supplies    719047.03     122490.80           13.80
```

> **Note:** Furniture pulls in $742K in revenue but only $18K in profit — the lowest margin at 3.88%. Office Supplies is surprisingly efficient by comparison.

In [12]:
# 4.3 — Sub-Category breakdown (Sales + Units)
q = """
SELECT Sub_Category,
       ROUND(SUM(Sales), 2) AS Total_Sales,
       SUM(Quantity)        AS Total_Units
FROM   superstore
GROUP  BY Sub_Category
ORDER  BY Total_Sales DESC
LIMIT  10;
"""
pd.read_sql_query(q, con)

,Sub_Category,Total_Sales,Total_Units
0,Phones,330007.05,3289
1,Chairs,328449.10,2356
2,Storage,223843.61,3158
3,Tables,206965.53,1241
4,Binders,203412.73,5974
5,Machines,189238.63,440
6,Accessories,167380.32,2976
7,Copiers,149528.03,234
8,Bookcases,114880.00,868
9,Appliances,107532.16,1729


**Output:**
```
Sub_Category  Total_Sales  Total_Units
      Phones    330007.05         3289
      Chairs    328449.10         2356
     Storage    223843.61         3158
      Tables    206965.53         1241
     Binders    203412.73         5974
    Machines    189238.63          440
 Accessories    167380.32         2976
     Copiers    149528.03          234
   Bookcases    114880.00          868
  Appliances    107532.16         1729
```

In [13]:
# 4.4 — Segment Summary with Avg Shipping Delay
q = """
SELECT Segment,
       ROUND(SUM(Sales), 2)          AS Total_Sales,
       ROUND(AVG(Ship_Delay_Days), 1) AS Avg_Ship_Days
FROM   superstore
GROUP  BY Segment;
"""
pd.read_sql_query(q, con)

,Segment,Total_Sales,Avg_Ship_Days
0,Consumer,1161401.34,3.9
1,Corporate,706146.37,4.0
2,Home Office,429653.15,3.9


**Output:**
```
    Segment  Total_Sales  Avg_Ship_Days
   Consumer   1161401.34            3.9
  Corporate    706146.37            4.0
Home Office    429653.15            3.9
```

In [14]:
# 4.5 — Discount Band Profitability (using engineered Discount_Band)
q = """
SELECT Discount_Band,
       COUNT(*)                        AS Orders,
       ROUND(SUM(Profit), 2)           AS Total_Profit,
       ROUND(AVG(Profit_Margin_Pct), 2) AS Avg_Margin
FROM   superstore
GROUP  BY Discount_Band
ORDER  BY Discount_Band;
"""
pd.read_sql_query(q, con)

,Discount_Band,Orders,Total_Profit,Avg_Margin
0,High,933,-99558.59,-108.90
1,Low,3803,100785.47,17.44
2,Medium,460,-35817.47,-16.69
3,No Discount,4798,320987.60,34.02


**Output:**
```
Discount_Band  Orders  Total_Profit  Avg_Margin
         High     933     -99558.59     -108.90
          Low    3803     100785.47       17.44
       Medium     460     -35817.47      -16.69
  No Discount    4798     320987.60       34.02
```

> **Critical finding:** Orders with High discount (>40%) generated a collective loss of nearly $100K. No-discount orders averaged a 34% margin.

---
## Step 5 — Sort and Limit (Top Products & Categories)

## Step 5 - Top Products and Categories

Here I sorted the data to find which products and sub-categories 
are generating the most revenue and profit. Used ORDER BY and LIMIT 
to get the top results.

In [15]:
# 5.1 — Top 5 Products by Revenue
q = """
SELECT Product_Name,
       ROUND(SUM(Sales), 2) AS Total_Sales
FROM   superstore
GROUP  BY Product_Name
ORDER  BY Total_Sales DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Product_Name,Total_Sales
0,Canon imageCLASS 2200 Advanced Copier,61599.82
1,Fellowes PB500 Electric Punch Plastic Comb Bin...,27453.38
2,Cisco TelePresence System EX90 Videoconferenci...,22638.48
3,HON 5400 Series Task Chairs for Big and Tall,21870.58
4,GBC DocuBind TL300 Electric Binding System,19823.48


**Output:**
```
                                              Product_Name  Total_Sales
                   Canon imageCLASS 2200 Advanced Copier     61599.82
Fellowes PB500 Electric Punch Plastic Comb Binding Machine   27453.38
       Cisco TelePresence System EX90 Videoconferencing Unit  22638.48
            HON 5400 Series Task Chairs for Big and Tall       21870.58
              GBC DocuBind TL300 Electric Binding System       19823.48
```

In [16]:
# 5.2 — Top 5 Sub-Category + Category combinations by Profit
q = """
SELECT Category,
       Sub_Category,
       ROUND(SUM(Profit), 2) AS Total_Profit
FROM   superstore
GROUP  BY Category, Sub_Category
ORDER  BY Total_Profit DESC
LIMIT  5;
"""
pd.read_sql_query(q, con)

,Category,Sub_Category,Total_Profit
0,Technology,Copiers,55617.82
1,Technology,Phones,44515.73
2,Technology,Accessories,41936.64
3,Office Supplies,Paper,34053.57
4,Office Supplies,Binders,30221.76


**Output:**
```
        Category Sub_Category  Total_Profit
      Technology      Copiers      55617.82
      Technology       Phones      44515.73
      Technology  Accessories      41936.64
 Office Supplies        Paper      34053.57
 Office Supplies      Binders      30221.76
```

---
## Step 6 — Use Cases

Here I looked at monthly sales trends, top customers by revenue and also 
checked for duplicate records in the data.

In [17]:
# 6.1 — Monthly Sales and Profit Trend
q = """
SELECT strftime('%Y-%m', Order_Date) AS Month,
       ROUND(SUM(Sales), 2)  AS Monthly_Sales,
       ROUND(SUM(Profit), 2) AS Monthly_Profit
FROM   superstore
GROUP  BY Month
ORDER  BY Month;
"""
monthly = pd.read_sql_query(q, con)
print(monthly.to_string(index=False))

  Month  Monthly_Sales  Monthly_Profit
2014-01       14236.90         2450.19
2014-02        4519.89          862.31
2014-03       55691.01          498.73
2014-04       28295.35         3488.84
2014-05       23648.29         2738.71
2014-06       34595.13         4976.52
2014-07       33946.39         -841.48
2014-08       27909.47         5318.10
2014-09       81777.35         8328.10
2014-10       31453.39         3448.26
2014-11       78628.72         9292.13
2014-12       69545.62         8983.57
2015-01       18174.08        -3281.01
2015-02       11951.41         2813.85
2015-03       38726.25         9732.10
2015-04       34195.21         4187.50
2015-05       30131.69         4667.87
2015-06       24797.29         3335.56
2015-07       28765.33         3288.65
2015-08       36898.33         5355.81
2015-09       64595.92         8209.16
2015-10       31404.92         2817.37
2015-11       75972.56        12474.79
2015-12       74919.52         8016.97
2016-01       18542.49   

**Partial Output (2017):**
```
   Month  Monthly_Sales  Monthly_Profit
 2017-01       43971.37         7140.44
 2017-02       20301.13         1613.87
 2017-03       58872.35        14751.89
 2017-04       36521.54          933.29
 2017-05       44261.11         6342.58
 2017-06       52981.73         8223.34
 2017-07       45264.42         6952.62
 2017-08       63120.89         9040.96
 2017-09       87866.65        10991.56
 2017-10       77776.92         9275.28
 2017-11      118447.82         9690.10
 2017-12       83829.32         8483.35
```

> **Observation:** November 2017 peaks at $118K in sales — a clear holiday/seasonal effect. Q4 consistently outperforms the rest of the year across all years in the dataset.

In [18]:
# 6.2 — Top 10 Customers by Revenue
q = """
SELECT Customer_Name,
       Segment,
       ROUND(SUM(Sales), 2)           AS Total_Sales,
       ROUND(SUM(Profit), 2)          AS Total_Profit,
       COUNT(DISTINCT Order_ID)       AS Total_Orders
FROM   superstore
GROUP  BY Customer_Name
ORDER  BY Total_Sales DESC
LIMIT  10;
"""
pd.read_sql_query(q, con)

,Customer_Name,Segment,Total_Sales,Total_Profit,Total_Orders
0,Sean Miller,Home Office,25043.05,-1980.74,5
1,Tamara Chand,Corporate,19052.22,8981.32,5
2,Raymond Buch,Consumer,15117.34,6976.10,6
3,Tom Ashbrook,Home Office,14595.62,4703.79,4
4,Adrian Barton,Consumer,14473.57,5444.81,10
5,Ken Lonsdale,Consumer,14175.23,806.85,12
6,Sanjit Chand,Consumer,14142.33,5757.41,9
7,Hunter Lopez,Consumer,12873.30,5622.43,6
8,Sanjit Engle,Consumer,12209.44,2650.68,11
9,Christopher Conant,Consumer,12129.07,2177.05,5


**Output:**
```
     Customer_Name     Segment  Total_Sales  Total_Profit  Total_Orders
       Sean Miller Home Office     25043.05      -1980.74             5
      Tamara Chand   Corporate     19052.22       8981.32             5
      Raymond Buch    Consumer     15117.34       6976.10             6
      Tom Ashbrook Home Office     14595.62       4703.79             4
     Adrian Barton    Consumer     14473.57       5444.81            10
      Ken Lonsdale    Consumer     14175.23        806.85            12
      Sanjit Chand    Consumer     14142.33       5757.41             9
      Hunter Lopez    Consumer     12873.30       5622.43             6
      Sanjit Engle    Consumer     12209.44       2650.68            11
Christopher Conant    Consumer     12129.07       2177.05             5
```

> **Note:** Sean Miller is the #1 customer by revenue ($25K) but is actually unprofitable at -$1,980. Ken Lonsdale placed 12 orders but only generated $807 in profit — potential discount abuse.

In [19]:
# 6.3 — Detect Duplicate Line Items (same Order + same Product)
q = """
SELECT Order_ID, Product_ID, COUNT(*) AS Freq
FROM   superstore
GROUP  BY Order_ID, Product_ID
HAVING Freq > 1;
"""
pd.read_sql_query(q, con)

,Order_ID,Product_ID,Freq
0,CA-2015-103135,OFF-BI-10000069,2
1,CA-2016-129714,OFF-PA-10001970,2
2,CA-2016-137043,FUR-FU-10003664,2
3,CA-2016-140571,OFF-PA-10001954,2
4,CA-2017-118017,TEC-AC-10002006,2
5,CA-2017-152912,OFF-ST-10003208,2
6,US-2014-150119,FUR-CH-10002965,2
7,US-2016-123750,TEC-AC-10004659,2


**Output:**
```
      Order_ID      Product_ID  Freq
CA-2015-103135 OFF-BI-10000069     2
CA-2016-129714 OFF-PA-10001970     2
CA-2016-137043 FUR-FU-10003664     2
CA-2016-140571 OFF-PA-10001954     2
CA-2017-118017 TEC-AC-10002006     2
CA-2017-152912 OFF-ST-10003208     2
US-2014-150119 FUR-CH-10002965     2
US-2016-123750 TEC-AC-10004659     2
```

> **Finding:** 8 (Order, Product) pairs appear twice — likely split shipments or data entry errors. These should be investigated before any revenue rollup.

In [20]:
# 6.4 — Regional Efficiency (Margin, Rev/Unit, Shipping Speed)
q = """
SELECT Region,
       ROUND(AVG(Profit_Margin_Pct), 2)  AS Avg_Margin_Pct,
       ROUND(AVG(Revenue_per_Unit), 2)   AS Avg_Rev_Per_Unit,
       ROUND(AVG(Ship_Delay_Days), 1)    AS Avg_Ship_Days
FROM   superstore
GROUP  BY Region
ORDER  BY Avg_Margin_Pct DESC;
"""
pd.read_sql_query(q, con)

,Region,Avg_Margin_Pct,Avg_Rev_Per_Unit,Avg_Ship_Days
0,West,21.95,60.70,3.9
1,East,16.72,64.09,3.9
2,South,16.35,61.69,4.0
3,Central,-10.41,56.79,4.1


**Output:**
```
 Region  Avg_Margin_Pct  Avg_Rev_Per_Unit  Avg_Ship_Days
   West           21.95             60.70            3.9
   East           16.72             64.09            3.9
  South           16.35             61.69            4.0
Central          -10.41             56.79            4.1
```

> **Alarming:** Central region has a negative average margin (-10.41%). Despite comparable shipping delays, the discount strategy in Central appears severely eroding profitability.

---
## Step 7 — Data Validation

Before finalizing I wanted to make sure the data is clean. Checked for 
null values, zero sales and verified total row count matches the original file.

In [21]:
# 7.1 — Basic data quality checks
q = """
SELECT COUNT(*)                                     AS Total_Rows,
       SUM(CASE WHEN Sales <= 0 THEN 1 ELSE 0 END)  AS Zero_Or_Neg_Sales,
       SUM(CASE WHEN Profit IS NULL THEN 1 ELSE 0 END) AS Null_Profit,
       MIN(Sales)                                   AS Min_Sales,
       MAX(Sales)                                   AS Max_Sales
FROM   superstore;
"""
pd.read_sql_query(q, con)

,Total_Rows,Zero_Or_Neg_Sales,Null_Profit,Min_Sales,Max_Sales
0,9994,0,0,0.444,22638.48


**Output:**
```
 Total_Rows  Zero_Or_Neg_Sales  Null_Profit  Min_Sales  Max_Sales
       9994                  0            0      0.444   22638.48
```

> Dataset is clean — no nulls in Profit, no zero or negative Sales values. Row count matches the raw CSV.

In [22]:
# 7.2 — Cross-check total sales and profit vs expected
q = """
SELECT ROUND(SUM(Sales), 2)  AS Grand_Total_Sales,
       ROUND(SUM(Profit), 2) AS Grand_Total_Profit,
       ROUND(SUM(Profit) * 100.0 / SUM(Sales), 2) AS Overall_Margin_Pct
FROM   superstore;
"""
pd.read_sql_query(q, con)

,Grand_Total_Sales,Grand_Total_Profit,Overall_Margin_Pct
0,2297200.86,286397.02,12.47


**Output:**
```
 Grand_Total_Sales  Grand_Total_Profit  Overall_Margin_Pct
        2297200.86          286397.02               12.47
```

> Total revenue: **$2.3M** | Total profit: **$286K** | Overall margin: **12.47%**

In [23]:
con.close()
print('Connection closed.')

Connection closed.


---
## Summary of Key Business Insights

| Area | Finding |
|---|---|
| **Revenue leader** | West region ($725K), Technology category ($836K) |
| **Profit leader** | Technology at 15.6% avg margin; Copiers sub-cat alone = $55K profit |
| **Biggest risk** | Furniture's 3.9% margin and Central region's **negative** average margin (-10.4%) |
| **Discount impact** | High-discount (>40%) orders collectively lost $99K — discounting policy needs a ceiling |
| **Top customer** | Sean Miller leads in revenue ($25K) but is net-unprofitable (-$1,980) |
| **Seasonality** | Clear Q4 sales spike every year; November 2017 peaked at $118K |
| **Data issues** | 8 duplicate (Order, Product) pairs — worth investigating for split-shipment errors |
| **Shipping** | Central region averages 4.1 days to ship vs 3.9 in West/East — slightly slower |